In [ ]:
import os
from pathlib import Path
DATA = Path(os.environ.get('ENSO_DATA_ROOT', Path.cwd() / 'data'))

import itertools
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
import scipy.io as sio
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind   #双样本t检验
import colormaps as cmaps  # 确保 colormaps 库已安装
import matplotlib.colors as mcolors

In [ ]:

data_path_psi = str(DATA / "psi.mat")
lon = sio.loadmat(data_path_psi)['lon_need'].flatten()
plev = sio.loadmat(data_path_psi)['plev'].flatten().astype(np.int32)
diff = sio.loadmat(data_path_psi)['diff'].T
h = sio.loadmat(data_path_psi)['h'].T
psi_ave = sio.loadmat(data_path_psi)['psi_ave']
psi105 = psi_ave[:,:,0].T
psi285 = psi_ave[:,:,1].T
X, Y = np.meshgrid(lon,-plev)

In [ ]:
point = np.zeros((17*128, 4))
num = 0
for i in range(psi105.shape[0]):
    for j in range(psi105.shape[1]):
        if psi105[i, j] > 150:
            point[num, 0] = lon[j]  # 将经度存储到第一列
            point[num, 1] = plev[i]  # 将气压存储到第二列
            point[num, 2] = psi105[i, j]  # 将 psi 值存储到第三列
            num += 1
# 截取有效的数据
point = point[:num]  # 只保留有效的数据
# 计算经度加权平均
weights_x = point[:, 0] * point[:, 2]  # 使用经度和 psi 值计算权重
total_weight_x = sum(point[:, 2])  # 计算总权重
centroid_x = sum(weights_x) / total_weight_x

weights_y = point[:, 1] * point[:, 2]  # 使用经度和 psi 值计算权重
total_weight_y = sum(point[:, 2])  # 计算总权重
centroid_y = sum(weights_y) / total_weight_y

print("经度加权平均:", centroid_x)
print("高度加权平均:", centroid_y)
x105=centroid_x
y105=centroid_y


In [ ]:
point = np.zeros((17*128, 4))
num = 0
for i in range(psi285.shape[0]):
    for j in range(psi285.shape[1]):
        if psi285[i, j] > 150:
            point[num, 0] = lon[j]  # 将经度存储到第一列
            point[num, 1] = plev[i]  # 将气压存储到第二列
            point[num, 2] = psi285[i, j]  # 将 psi 值存储到第三列
            num += 1
# 截取有效的数据
point = point[:num]  # 只保留有效的数据
# 计算经度加权平均
weights_x = point[:, 0] * point[:, 2]  # 使用经度和 psi 值计算权重
total_weight_x = sum(point[:, 2])  # 计算总权重
centroid_x = sum(weights_x) / total_weight_x

weights_y = point[:, 1] * point[:, 2]  # 使用经度和 psi 值计算权重
total_weight_y = sum(point[:, 2])  # 计算总权重
centroid_y = sum(weights_y) / total_weight_y

print("经度加权平均:", centroid_x)
print("高度加权平均:", centroid_y)
x285=centroid_x
y285=centroid_y


In [ ]:
#绘图
fig = plt.figure(figsize=(10,10))
plt.rcParams['figure.figsize'] = (16,4)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'normal' 
plt.rcParams.update({'font.size': 16})

gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1])  # 调整子图高度
plt.subplots_adjust(hspace=0.2)  # 调整子图间隔

plt.subplot(311)
# 填充等高线
norm = mcolors.TwoSlopeNorm(vmin=-270, vcenter=0, vmax=150)

# 然后画图
c1 = plt.contourf(X, Y, psi105, 
                  levels=np.arange(-270, 150, 0.1), 
                  cmap=custom_cmap, 
                  norm=norm, 
                  extend='both')


h1 = plt.contour(X, Y, psi105, levels=np.arange(60,180,30), colors='k', linewidths=1)
plt.clabel(h1, inline=True, fontsize=11, fmt='%d')
h2 = plt.contour(X, Y, psi105, levels=np.arange(-300,-60,30), colors='k', linestyles='-.', linewidths=1)
plt.clabel(h2, inline=True, fontsize=11, fmt='%d')
h3 = plt.contour(X, Y, psi105, levels=[0], colors='k', linewidths=2)
plt.clabel(h3, inline=True, fontsize=11, fmt='%d')

plt.xlim([40, 280])
plt.xticks(np.arange(40,300, 20),['40°E','60°E', '80°E','100°E' ,'120°E',  '140°E', '160°E','180°', '160°W', '140°W', '120°W', '100°W','80°W'], rotation=0, fontsize=11, fontweight='normal')
plt.tick_params(axis='x', labelbottom=False)  # 隐藏x轴刻度标签
plt.ylim([-1000, -100])
plt.yticks(np.arange(-1000, -99, 100),
                ['1000', '', '', '700', '', '', '400', '', '', '100'],
                fontsize=11, fontweight='normal')

plt.ylabel('Pressure [hPa]', fontsize=11, color='k', fontweight='normal')
# plt.grid(True, linestyle='-.', color='lightgray')


plt.text(40,-50, '(a) Precessional Phase 90°', color='black', fontsize=11, weight='normal')


plt.subplot(312)
# 填充等高线
c2 = plt.contourf(X, Y, psi285, 
                  levels=np.arange(-270, 150, 0.1), 
                  cmap=custom_cmap, 
                  norm=norm, 
                  extend='both')
h4 = plt.contour(X, Y, psi285, levels=np.arange(60,180,30), colors='k', linewidths=1)
plt.clabel(h4, inline=True, fontsize=11, fmt='%d')
h5 = plt.contour(X, Y, psi285, levels=np.arange(-300,-60,30), colors='k', linestyles='-.', linewidths=1)
plt.clabel(h5, inline=True, fontsize=11, fmt='%d')
h6 = plt.contour(X, Y, psi285, levels=[0], colors='k', linewidths=2)
plt.clabel(h6, inline=True, fontsize=11, fmt='%d')


plt.xlim([40, 280])
plt.xticks(np.arange(40,300, 20),['40°E','60°E', '80°E','100°E' ,'120°E',  '140°E', '160°E','180°', '160°W', '140°W', '120°W', '100°W','80°W'], rotation=0, fontsize=11, fontweight='normal')
plt.ylim([-1000, -100])
plt.tick_params(axis='x', labelbottom=False)  # 隐藏x轴刻度标签
plt.yticks(np.arange(-1000, -99, 100),
                ['1000', '', '', '700', '', '', '400', '', '', '100'],
                fontsize=11, fontweight='normal')

plt.ylabel('Pressure [hPa]', fontsize=11, color='k', fontweight='normal')
# plt.grid(True, linestyle='-.', color='lightgray')
plt.text(40,-50, '(b) Precessional Phase 270°', color='black', fontsize=11, weight='normal')


plt.subplot(313)
# 填充等高线
c3 = plt.contourf(X, Y, diff, levels=np.arange(-180,180, 0.01), cmap=cmaps.div1_blue_orange, extend='both')

h1 = plt.contour(X, Y, psi105, levels=np.arange(60,180,30), colors='k', linewidths=1)
plt.clabel(h1, inline=True, fontsize=11, fmt='%d')
h2 = plt.contour(X, Y, psi105, levels=np.arange(-270,-60,30), colors='k', linestyles='-.', linewidths=1)
plt.clabel(h2, inline=True, fontsize=11, fmt='%d')
h3 = plt.contour(X, Y, psi105, levels=[0], colors='k', linewidths=2)
plt.clabel(h3, inline=True, fontsize=11, fmt='%d')

# # 对置信区间打点   找到值为 1 的点的位置索引
plevh, lonh = np.where(h== 1)
a=lon[lonh]
b=-plev[plevh]
plt.scatter(a, b,color='lightgray', s=0.1, marker='p', label='显著性点', alpha=0.8)  # 浅灰色

# 设置坐标轴标签和刻度
plt.xlabel('Longitude', fontsize=11, color='k', fontweight='normal')
plt.xlim([40, 280])
plt.xticks(np.arange(40,300, 20),['40°E','60°E', '80°E','100°E' ,'120°E',  '140°E', '160°E','180°', '160°W', '140°W', '120°W', '100°W','80°W'], rotation=0,
                fontsize=11, fontweight='normal')
plt.ylim([-1000, -100])
plt.yticks(np.arange(-1000, -99, 100),
                ['1000', '', '', '700', '', '', '400', '', '', '100'],
                fontsize=11, fontweight='normal')
plt.ylabel('Pressure [hPa]', fontsize=11, color='k', fontweight='normal')
plt.legend

plt.text(40,-50, '(c) Anomaly', color='black', fontsize=11 ,weight='normal')



# 添加colorbar
cax1 = fig.add_axes([0.92, 0.41, 0.015, 0.45])  # 这四个数可以慢慢试
colorbar1 = fig.colorbar(c1, cax=cax1, orientation='vertical')
colorbar1.set_label('Streamfunction [10$^6$ kg/s]', fontsize=11, fontweight='normal')
ticks = np.arange(-280,141,70)
colorbar1.set_ticks(ticks)
ticklabels = [int(tick) for tick in ticks]
colorbar1.set_ticklabels(ticklabels, fontsize=11, fontweight='normal')



# 添加colorbar
cax2 = fig.add_axes([0.92, 0.12, 0.015, 0.2])  # 这四个数可以慢慢试
colorbar1 = fig.colorbar(c3, cax=cax2, orientation='vertical')
colorbar1.set_label('Streamfunction Anomaly [10$^6$ kg/s]', fontsize=11, fontweight='normal')
ticks = np.arange(-160,161,80)
colorbar1.set_ticks(ticks)
ticklabels = [int(tick) for tick in ticks]
colorbar1.set_ticklabels(ticklabels, fontsize=11, fontweight='normal')


plt.rcParams.update({'font.size': 11})
plt.savefig('walker_supply.png') 
plt.show()

In [ ]:
# 添加图例
colorbar = fig.colorbar(c1, ax=plt.gca(), orientation='vertical', pad=0.02)
colorbar.set_label('[×10^6 kg/s]', fontsize=11, fontweight='normal')
ticks = np.arange(-250,200,50)
colorbar.set_ticks(ticks)
ticklabels = [int(tick) for tick in ticks]
colorbar.set_ticklabels(ticklabels, fontsize=11, fontweight='normal')
colorbar.ax.set_position([0.78, 0.655, 0.015, 0.22])


In [ ]:
from matplotlib.colors import ListedColormap
# 加载 .rgb 文件中的颜色代码
colorcode = np.loadtxt(str(DATA / "colormaps" / "testcmap (2).rgb"))
colorcode = colorcode /256
# 创建并测试自定义色带
custom_cmap= ListedColormap(colorcode)
data = np.random.rand(8, 8)
plt.imshow(data, cmap=custom_cmap)
plt.colorbar()
plt.show()